# Objective

This notebook builds baseline models using a small synthetic feature set. It contains a rule-based baseline, a Logistic Regression baseline, and a Random Forest baseline.

# Baseline 1: Rule-based model

The rule-based baseline predicts decline when position worsens or clicks fall below a threshold.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/seo_content_performance.csv')

rule_pred = np.where((df['position_30d'] > df['position_7d'] + 4) | (df['clicks_30d'] < df['clicks_7d'] * 0.8), 1, 0)
print('rule baseline predictions:', int(rule_pred.sum()), 'declines')

# Baseline 2: Logistic Regression

Logistic Regression is selected because it provides a simple interpretable linear baseline.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

features = [
    'content_type', 'category', 'word_count', 'content_age_days', 'author_type', 'has_schema',
    'internal_link_count', 'external_link_count', 'organic_clicks', 'organic_impressions',
    'ctr', 'average_position', 'keyword_count', 'ranking_keywords', 'backlinks',
    'domain_authority', 'sessions', 'bounce_rate', 'avg_session_duration', 'conversions',
    'clicks_7d', 'clicks_30d', 'impressions_7d', 'impressions_30d', 'position_7d', 'position_30d'
]
X = df[features]
y = df['is_declining_label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = X.select_dtypes(include='number').columns
cat_cols = [c for c in X.columns if c not in num_cols]
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])
model = Pipeline([('prep', pre), ('clf', LogisticRegression(max_iter=500, random_state=42))])
model.fit(X_train, y_train)
print('Logistic Regression baseline trained')

# Baseline 3: Random Forest

Random Forest captures nonlinear relationships and provides a feature importance table.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([('prep', pre), ('clf', RandomForestClassifier(n_estimators=50, random_state=42))])
rf.fit(X_train, y_train)
print('Random Forest baseline trained')

# Results

The goal of this notebook is to understand the baseline behavior before the leakage investigation.